# Lab | Tools prompting

**Replace the existing two tools decorators, by creating 3 new ones and adjust the prompts accordingly**

### How to add ad-hoc tool calling capability to LLMs and Chat Models

:::{.callout-caution}

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

## Setup

We'll need to install the following packages:

In [ ]:
#%pip install --upgrade --quiet langchain langchain-community langchain_openai

In [1]:
# Version Verification

%pip show langchain langchain-community langchain-openai langchain-core ipykernel | grep -E "^(Name|Version):"

Name: langchain
Version: 1.3.2
Name: langchain-community
Version: 0.4.2
Name: langchain-openai
Version: 1.2.2
Name: langchain-core
Version: 1.4.0
Name: ipykernel
Version: 7.2.0
Note: you may need to restart the kernel to use updated packages.


If you'd like to use LangSmith, uncomment the below:

In [1]:
import getpass
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

```{=mdx}
import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs openaiParams={`model="gpt-4"`} />
```

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [2]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")

/var/folders/dk/w5mlpx9x4t9fj30jk64759lh0000gn/T/ipykernel_13928/1427064109.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
/var/folders/dk/w5mlpx9x4t9fj30jk64759lh0000gn/T/ipykernel_13928/1427064109.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  model = Ollama(model="phi3")



#  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

## Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download)  
   Download and run the installer.

2. **Verify Installation**  
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**  
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**  
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [4]:
from langchain_core.tools import tool


@tool
def co2_emissions(distance: float, factor: float) -> float:
    """Calculate CO2 emissions: distance × emission factor."""
    return round(distance * factor, 2)
#def multiply(x: float, y: float) -> float:
#    """Multiply two numbers together. Show the numbers in the answer and also poit #if they are positive, negative, float or integers"""
#    return x * y


@tool
def trees_needed(co2_kg: float, avg_tree_co2: float = 21.0) -> float:
    """Calculate how many trees needed to offset CO2 (default: 21 kg/year per tree)."""
    return round(co2_kg / avg_tree_co2, 1)

#def add(x: int, y: int) -> int:
#    "Add two numbers. "
#    return x + y

@tool
def temp_increase(years: int, rate: float = 0.03) -> float:
    """Project temperature increase: years × annual rate (default: 0.03°C/year)."""
    return round(years * rate, 2)

tools = [co2_emissions, trees_needed, temp_increase]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
co2_emissions
Calculate CO2 emissions: distance × emission factor.
{'distance': {'title': 'Distance', 'type': 'number'}, 'factor': {'title': 'Factor', 'type': 'number'}}
--
trees_needed
Calculate how many trees needed to offset CO2 (default: 21 kg/year per tree).
{'co2_kg': {'title': 'Co2 Kg', 'type': 'number'}, 'avg_tree_co2': {'default': 21.0, 'title': 'Avg Tree Co2', 'type': 'number'}}
--
temp_increase
Project temperature increase: years × annual rate (default: 0.03°C/year).
{'years': {'title': 'Years', 'type': 'integer'}, 'rate': {'default': 0.03, 'title': 'Rate', 'type': 'number'}}


In [ ]:
#Try with mathematic example
#multiply.invoke({"x": 4, "y": 5})

20.0

In [6]:
print("CO2 from a 400km flight:", co2_emissions.invoke({"distance": 400, "factor": 0.192}))
print("Trees to offset 76.8 kg:", trees_needed.invoke({"co2_kg": 76.8}))
print("Increase in temperature in 30 years:", temp_increase.invoke({"years": 30}))

CO2 from a 400km flight: 76.8
Trees to offset 76.8 kg: 3.7
Increase in temperature in 30 years: 0.9


## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [7]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

co2_emissions(distance: float, factor: float) -> float - Calculate CO2 emissions: distance × emission factor.
trees_needed(co2_kg: float, avg_tree_co2: float = 21.0) -> float - Calculate how many trees needed to offset CO2 (default: 21 kg/year per tree).
temp_increase(years: int, rate: float = 0.03) -> float - Project temperature increase: years × annual rate (default: 0.03°C/year).


In [8]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [9]:
chain = prompt | model
message = chain.invoke({"input": "I want to travel 600k by plane this month. What could be my CO2 print?"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

```json
{
  "name": "co2_emissions",
  "arguments": {
    "distance": 600,
    "factor": 0decibel_value # Assuming an emission factor (kg of CO2 per km) is provided or available for calculation.
  }
}
```
Remember to replace `decibel_value` with the actual CO2 emissions factor specific for air travel used in your calculations, which can vary by aircraft type and fuel efficiency among other factors. If not explicitly mentioned, a typical value might be around 135g/km or about 0.135 kg/km when converted to kilogranes per kilometer (kg/km).


## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [11]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()

chain.invoke({"input":"How many trees do I need to offset 100 kg of CO2"})
#chain.invoke({"input": "what's thirteen times 4"})

{'name': 'trees_needed', 'arguments': {'co2_kg': 100, 'avg_tree_co2': 21.0}}

:::{.callout-important}

🎉 Amazing! 🎉 We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!
:::

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke 
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [12]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

Let's test this out 🧪!

In [ ]:
#Try with the older math tools
#invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})

15.0

In [16]:
#Tool 1
print("CO2:", invoke_tool({"name": "co2_emissions", "arguments": {"distance": 400, "factor": 0.192}}))
print("Trees:", invoke_tool({"name": "trees_needed", "arguments": {"co2_kg": 76.8}}))
print("Temperature:", invoke_tool({"name": "temp_increase", "arguments": {"years": 30}}))

CO2: 76.8
Trees: 3.7
Temperature: 0.9


## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [ ]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "what's the increment of temperature in 4 years?"})
#chain.invoke({"input": "what's thirteen times 4.14137281"})

0.0

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [20]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "what's the CO2 emission for 600 km by plane? Use factor 0.255"})

#chain.invoke({"input": "what's thirteen times 4.14137281"})

{'name': 'co2_emissions',
 'arguments': {'distance': 600, 'factor': 0},
 'output': 0.0}

## What's next?

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information.

In reality, if you're using more complex tools, you will start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

You will need to be prepared to add strategies to improve the output from the model; e.g.,

1. Provide few shot examples.
2. Add error handling (e.g., catch the exception and feed it back to the LLM to ask it to correct its previous output).

In [21]:
few_shot_examples = """
Here are some examples of correct responses:

User: "I flew 1200 km last week"
Response: {"name": "co2_emissions", "arguments": {"distance": 1200, "factor": 0.255}}

User: "How many trees to offset 150 kg of CO2?"
Response: {"name": "trees_needed", "arguments": {"co2_kg": 150}}

User: "Project temperature rise over 25 years"
Response: {"name": "temp_increase", "arguments": {"years": 25}}
"""

system_prompt = f"""\
You are a climate assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

{few_shot_examples}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.
All argument values MUST be numbers — never leave placeholders or comments.
Respond ONLY with the JSON, no explanations.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.
"""

In [22]:
from langchain_core.exceptions import OutputParserException
import json

def chain_with_retry(user_input: str, max_retries: int = 2):
    """Invoke the chain with automatic retry on JSON errors."""
    last_error = None
    
    for attempt in range(max_retries + 1):
        try:
            result = chain.invoke({"input": user_input})
            print(f"Success (attempt {attempt + 1})")
            return result
        
        except OutputParserException as e:
            last_error = e
            if attempt == max_retries:
                break
            
            # Feed the error back to the model
            correction_prompt = f"""\
Your previous response had invalid JSON. Error: {str(e)}

User asked: "{user_input}"

Return ONLY a valid JSON blob with 'name' and 'arguments'. No explanations, no placeholders.
Example: {{"name": "co2_emissions", "arguments": {{"distance": 600, "factor": 0.255}}}}
"""
            print(f"Retry {attempt + 1}: model returned bad JSON, asking to fix...")
            user_input = correction_prompt
    
    print(f"Failed after {max_retries + 1} attempts: {last_error}")
    return None

In [27]:
print("--- Test 1: co2_emissions ---")
print(chain_with_retry("What's the CO2 for a 600 km flight? Use factor 1.255"))



--- Test 1: co2_emissions ---
Success (attempt 1)
{'name': 'co2_emissions', 'arguments': {'distance': 600, 'factor': 1.255}, 'output': 753.0}


In [24]:
print("\n--- Test 2: trees_needed ---")
print(chain_with_retry("How many trees to offset 150 kg of CO2?"))



--- Test 2: trees_needed ---
Success (attempt 1)
{'name': 'trees_needed', 'arguments': {'co2_kg': 150, 'avg_tree_co2': 21.0}, 'output': 7.1}


In [26]:
print("\n--- Test 3: temp_increase ---")
print(chain_with_retry("Project temperature rise over 25 years"))


--- Test 3: temp_increase ---
Success (attempt 1)
{'name': 'temp_increase', 'arguments': {'years': 25, 'rate': 0}, 'output': 0.0}
